# Advanced Crypto Price Prediction Dashboard — v3.7

**Coins monitored:** BTC, SOL, ATOM, CRV, FET, BONK, WIF (configurable)

## Signal layers

| Layer | Signals | Why it helps |
|---|---|---|
| **Momentum** | RSI, MACD, Stochastic | Core price momentum |
| **Trend** | EMA alignment (9/20/50/100/200), ADX, VWAP, slopes | Distinguishes real moves from noise |
| **Volume** | OBV trend, buy/sell pressure, vol momentum | Confirms price moves with flow |
| **Volatility** | ATR percentile, BB squeeze, low-volume bullish penalty | Anticipates explosive moves, filters false breakouts |
| **Correlation** | Rolling BTC corr, BTC dominance trend, alt beta | Crypto doesn't move in isolation |
| **Sentiment** | Fear & Greed (extreme readings), CryptoPanic news | Markets are driven by emotion and headlines |
| **Funding Rates** *(new)* | CoinGlass long/short bias, funding rate direction | Detects over-leveraged rallies and squeeze setups |
| **Multi-Timeframe** *(new)* | 1h + 4h + 1d alignment check | Filters signals that only appear on one timeframe |
| **BTC Dominance** *(new)* | Dominance trend suppresses alt bullish signals | Capital rotation indicator |

## v3.7 (April 21 2026)
- **Section 14 copy button**: replaces plain print with a `📋 Copy to clipboard` button via ipywidgets. The analysis text appears in a scrollable monospace textarea with a green copy button above it. One click copies everything to clipboard ready to paste to Claude. Falls back to plain text print if ipywidgets unavailable.

## v3.6 (April 21 2026)
- **Section 14 — Copy for Analysis**: new cell at the end of the notebook that prints a compact, structured summary between `=== COPY START ===` and `=== COPY END ===` markers. Run after Section 10. Paste only that block to Claude for analysis — no need to copy full Jupyter output.

## v3.5 (April 20 2026)
- **Volatility scorer fixed**: BB squeeze (low bandwidth) was scoring as bearish (12-26) because bb_pct < 0.5 returned a low score. A squeeze means price is *coiling* — direction unknown. Active squeezes (bb_bw_pct < 20) now return 50 ± small nudge. Post-squeeze (20-40) returns half-weight. This removes volatility as a false key driver.
- **MTF bearish threshold tightened**: MTF now requires score ≤20 (not ≤45) to count as 'bearish' in the agreement count. The floor of 10 represents fully-bearish 1h — treating it the same as score=45 (barely bearish) was inflating bearish counts by 1-2 on every run.

## v3.4 (April 20 2026)
- **VERSION constant**: single source of truth in config — version now auto-updates in dashboard header
- **1h MTF floor raised 1→10**: score of 1 was triggering HIGH divergence on every coin with a bearish short-term signal, diluting the flag. 10 = confirmed bearish but not extreme
- **Divergence threshold raised 70→80pt**: reduces noise from routine 1h pullbacks. MOD threshold raised 55→60pt
- **1h sole-driver note**: when MTF is the only bearish sub-score, divergence flag now appends '(1h pullback only)' or '— 1h weak, watch for recovery' to distinguish from genuine multi-factor conflicts

## v3.3 (April 11 2026)
- **Divergence flag**: new column showing when sub-scores conflict strongly (e.g. FET April 11: funding=65 vs momentum=21, volatility=10). ⚡ HIGH = spread ≥70pts with 2+ bull AND 2+ bear subs — do not act on composite. ↕ MOD = spread ≥55pts — treat with caution.

## v3.2 fixes (April 11 2026)
- **Timezone changed** UTC → CET (Europe/Zurich)
- **Score convergence fix**: vol cap now scales proportionally so coins differentiate across 55-71 rather than all hitting 64

- **VOL_CAP_STRONG tightened** 0.50→0.65x: ATOM at 0.58x was slipping through to Strong Bullish
- **MTF zero bug fixed**: WIF/SOL returning 0/0 — now defaults to 50 (neutral) not 0
- **BONK funding fixed**: tries `1000BONK/USDT:USDT` format before marking unavailable
- **Funding conflict flag**: bullish signal with bearish funding now shows ⚠️funding in Key Driver
- **Funding rates**: CoinGlass API — detects over-leveraged longs (false rally risk) and short squeezes
- **Multi-timeframe confirmation**: 1h + 4h + 1d must broadly agree — eliminates single-timeframe false signals
- **Low-volume bullish penalty**: Vol ratio < 0.7x now penalises bullish scores (prevented April 6 false 85/100)
- **BTC dominance trend**: Rising dominance suppresses alt bullish signals automatically
- **News sentiment**: CryptoPanic free API adds headline sentiment layer

**Horizon:** 3–10 day | **Horizon candles:** 4h primary, 1h + 1d confirmation

> ⚠️ This is structured analysis only. It does **not** provide financial advice or price targets. All models are wrong — some are useful.

## 1) Install & Imports

In [ ]:
# Uncomment to install dependencies
# !pip install -U ccxt pandas numpy requests scipy

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
from scipy import stats
from datetime import datetime, timezone
import time

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Imports OK")

## 2) Configuration

In [ ]:
# ─── Version ────────────────────────────────────────────────────────
VERSION = "3.7"   # bump here only — used in dashboard header automatically

# ─── Coins & Exchange ───────────────────────────────────────────────
COINS       = ["BTC", "SOL", "ATOM", "CRV", "FET", "BONK", "WIF"]
QUOTE       = "USDT"
EXCHANGE_ID = "binance"   # change to 'kraken', 'bybit', etc. if needed

# ─── Timeframes (multi-timeframe confirmation) ───────────────────────
TIMEFRAME        = "4h"    # primary timeframe
TF_FAST          = "1h"    # confirmation: faster
TF_SLOW          = "1d"    # confirmation: slower
LOOKBACK_BARS    = 360     # ~60 days of 4h bars
LOOKBACK_FAST    = 168     # ~7 days of 1h bars
LOOKBACK_SLOW    = 90      # 90 daily bars

# ─── Indicator Parameters ───────────────────────────────────────────
EMA_PERIODS   = [9, 20, 50, 100, 200]
RSI_PERIOD    = 14
MACD_FAST     = 12
MACD_SLOW     = 26
MACD_SIGNAL   = 9
BB_PERIOD     = 20
BB_STD        = 2.0
ADX_PERIOD    = 14
STOCH_K       = 14
STOCH_D       = 3
ATR_PERIOD    = 14
OBV_EMA_SPAN  = 20

# ─── Volume penalty threshold ────────────────────────────────────────
LOW_VOL_THRESHOLD = 0.7    # vol ratio below this penalises bullish scores

# ─── Hard signal cap thresholds (vol-based) ─────────────────────────
# Tightened April 9 (v3.1): ATOM at 0.58x slipped through at 0.50 threshold.
# Strong Bullish now requires vol > 0.65x; Bullish requires vol > 0.40x.
VOL_CAP_STRONG   = 0.65   # below this → max signal is 🟩 Bullish (cap at 71)
VOL_CAP_BULLISH  = 0.40   # below this → max signal is 🔆 Mildly Bullish (cap at 64)
# Bearish side: very low vol = don't allow Strong Bearish
VOL_CAP_BEAR     = 0.40   # below this → floor at 37

# ─── Scoring Weights (must sum to 1.0) ──────────────────────────────
# Reduced volatility weight (was 0.10) — it was dominating falsely
# Added funding (0.08) and mtf (0.07), reduced momentum slightly
WEIGHTS = {
    "momentum":    0.20,   # RSI, MACD, Stochastic
    "trend":       0.22,   # EMA alignment, ADX, price position
    "volume":      0.18,   # OBV, volume trend, buy pressure
    "volatility":  0.07,   # BB squeeze, ATR regime (reduced — was causing false signals)
    "correlation": 0.08,   # BTC correlation, dominance trend, alt beta
    "sentiment":   0.08,   # Fear & Greed + news sentiment
    "funding":     0.08,   # Funding rates (new)
    "mtf":         0.09,   # Multi-timeframe confirmation (new)
}

# ─── Signal Thresholds ──────────────────────────────────────────────
STRONG_SIGNAL_THRESHOLD = 65
WEAK_SIGNAL_THRESHOLD   = 35

# ─── API Keys (optional) ────────────────────────────────────────────
# CryptoPanic: free at https://cryptopanic.com/developers/api/
CRYPTOPANIC_KEY = ""      # leave blank to skip news sentiment

print(f"✅ Config v{VERSION}: {len(COINS)} coins | {TIMEFRAME} primary | {TF_FAST}/{TF_SLOW} confirmation")
print(f"   Weights: {WEIGHTS}")
print(f"   Sum of weights: {sum(WEIGHTS.values()):.2f} (must be 1.0)")

## 3) Data Fetching

In [ ]:
def fetch_ohlcv(symbol: str, timeframe: str = TIMEFRAME,
                limit: int = LOOKBACK_BARS,
                exchange_id: str = EXCHANGE_ID) -> pd.DataFrame:
    """Fetch OHLCV from CCXT with retry logic."""
    import ccxt
    ex_class = getattr(ccxt, exchange_id)
    ex = ex_class({"enableRateLimit": True})

    for attempt in range(3):
        try:
            ohlcv = ex.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
            df = pd.DataFrame(ohlcv, columns=["ts","open","high","low","close","volume"])
            df["ts"] = pd.to_datetime(df["ts"], unit="ms", utc=True)
            df = df.set_index("ts")
            df = df[df["volume"] > 0]
            return df
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f"Failed to fetch {symbol}: {e}")
            time.sleep(1.5)


def fetch_fear_greed() -> dict:
    """Fetch Crypto Fear & Greed Index from Alternative.me (free, no key)."""
    try:
        url = "https://api.alternative.me/fng/?limit=7&format=json"
        r = requests.get(url, timeout=8)
        data = r.json()["data"]
        current = int(data[0]["value"])
        label   = data[0]["value_classification"]
        avg7    = int(np.mean([int(d["value"]) for d in data]))
        trend   = "rising" if current > avg7 else "falling" if current < avg7 else "flat"
        return {"value": current, "label": label, "avg7": avg7, "trend": trend}
    except Exception:
        return {"value": 50, "label": "Neutral", "avg7": 50, "trend": "flat"}


def fetch_funding_rates(coins: list = COINS) -> dict:
    """
    Fetch funding rates from CoinGlass (free, no key needed for basic data).
    Returns dict: {coin: {rate: float, bias: str, score: float}}
    - Positive funding = longs paying shorts = overleveraged longs = bearish risk
    - Negative funding = shorts paying longs = overleveraged shorts = squeeze risk (bullish)
    - Near zero = balanced = neutral

    Falls back to Binance perpetual funding via CCXT if CoinGlass returns all zeros
    (a known silent failure mode observed April 8 2026).
    """
    result = {}

    def _rate_to_score(rate_pct: float, coin: str) -> tuple:
        if rate_pct < -0.05:
            return 80.0, "short squeeze risk"
        elif rate_pct < -0.01:
            return 65.0, "mild short bias"
        elif rate_pct <= 0.01:
            return 50.0, "neutral"
        elif rate_pct <= 0.05:
            return 38.0, "mild long overleverage"
        else:
            return 20.0, "crowded longs — bearish risk"

    # ── Try CoinGlass first ──────────────────────────────────────────
    coinglass_ok = False
    try:
        url = "https://open-api.coinglass.com/public/v2/funding"
        r = requests.get(url, timeout=10)
        data = r.json().get("data", [])
        lookup = {}
        for item in data:
            sym  = item.get("symbol", "").upper().replace("USDT","").replace("/","")
            rate = float(item.get("fundingRate", 0) or 0)
            lookup[sym] = rate

        # Check for all-zero response (silent failure)
        non_zero = sum(1 for v in lookup.values() if abs(v) > 1e-8)
        if non_zero > 0:
            coinglass_ok = True
            for coin in coins:
                rate     = lookup.get(coin, 0.0)
                rate_pct = rate * 100
                score, bias = _rate_to_score(rate_pct, coin)
                result[coin] = {"rate": round(rate_pct, 4), "bias": bias, "score": score,
                                "source": "coinglass"}
    except Exception:
        pass

    # ── Fallback: Binance perpetual funding via CCXT ─────────────────
    if not coinglass_ok:
        try:
            import ccxt
            ex = ccxt.binance({"enableRateLimit": True})
            # Some meme coins use different perp symbol formats on Binance
            # e.g. BONK trades as 1000BONK, WIF as WIF, FET as FET
            SYMBOL_OVERRIDES = {
                "BONK": ["1000BONK/USDT:USDT", "BONK/USDT:USDT"],
                "PEPE": ["1000PEPE/USDT:USDT", "PEPE/USDT:USDT"],
            }
            for coin in coins:
                sym_candidates = SYMBOL_OVERRIDES.get(coin, [f"{coin}/USDT:USDT"])
                fetched = False
                for sym in sym_candidates:
                    try:
                        info = ex.fetch_funding_rate(sym)
                        rate_pct = float(info.get("fundingRate", 0) or 0) * 100
                        score, bias = _rate_to_score(rate_pct, coin)
                        result[coin] = {"rate": round(rate_pct, 4), "bias": bias,
                                        "score": score, "source": "binance_perp"}
                        fetched = True
                        break
                    except Exception:
                        continue
                if not fetched:
                    result[coin] = {"rate": 0.0, "bias": "unavailable",
                                    "score": 50.0, "source": "none"}
        except Exception:
            for coin in coins:
                if coin not in result:
                    result[coin] = {"rate": 0.0, "bias": "unavailable",
                                    "score": 50.0, "source": "none"}

    return result


def fetch_btc_dominance() -> dict:
    """
    Fetch BTC dominance from CoinGecko (free, no key).
    Returns {dominance: float, trend: str}
    Rising dominance = bad for alts, good for BTC.
    """
    try:
        url = "https://api.coingecko.com/api/v3/global"
        r = requests.get(url, timeout=10)
        data = r.json()["data"]
        dom = float(data["market_cap_percentage"]["btc"])
        # CoinGecko doesn't provide historical in free tier
        # Use 50% as neutral baseline (historical BTC dom range: 38-65%)
        if dom > 58:
            trend = "rising_strong"   # very BTC-dominant
        elif dom > 52:
            trend = "rising"
        elif dom > 48:
            trend = "neutral"
        elif dom > 42:
            trend = "falling"         # alt season developing
        else:
            trend = "falling_strong"  # full alt season
        return {"dominance": round(dom, 2), "trend": trend}
    except Exception:
        return {"dominance": 50.0, "trend": "neutral"}


def fetch_news_sentiment(coins: list = COINS, api_key: str = CRYPTOPANIC_KEY) -> dict:
    """
    Fetch news sentiment from CryptoPanic API.
    Returns {coin: score} where score is 0-100.
    Falls back to 50 (neutral) if key not set or API fails.
    Get a free key at https://cryptopanic.com/developers/api/
    """
    result = {coin: 50.0 for coin in coins}
    if not api_key:
        return result
    try:
        for coin in coins:
            url = (f"https://cryptopanic.com/api/v1/posts/"
                   f"?auth_token={api_key}&currencies={coin}&filter=hot&public=true")
            r = requests.get(url, timeout=8)
            posts = r.json().get("results", [])
            if not posts:
                continue
            # Count bullish vs bearish votes in recent posts
            bull, bear, total = 0, 0, 0
            for post in posts[:20]:
                votes = post.get("votes", {})
                b = int(votes.get("positive", 0) or 0)
                bd = int(votes.get("negative", 0) or 0)
                bull += b
                bear += bd
                total += b + bd
            if total > 0:
                result[coin] = round((bull / total) * 100, 1)
            time.sleep(0.3)  # rate limit
    except Exception:
        pass
    return result


def fetch_mtf_signals(coin: str, quote: str = QUOTE,
                      exchange_id: str = EXCHANGE_ID) -> dict:
    """
    Fetch 1h and 1d OHLCV and compute simple trend signal for each.
    Returns {tf: score} where score is 0-100.

    v3.4 fixes:
    - Floor raised from 1 → 10 (score of 1 was treated as confirmed bearish,
      triggering HIGH divergence flags on every coin when 1h is weak)
    - Different minimum bar requirements per timeframe (1h needs 48+, 1d needs 20+)
    - Reliability flag: if fewer than required bars, stays at neutral 50
    """
    signals = {TF_FAST: 50.0, TF_SLOW: 50.0}   # safe default = neutral
    MIN_BARS = {TF_FAST: 48, TF_SLOW: 20}        # 1h: 2 days min, 1d: 20 days min
    sym = f"{coin}/{quote}"
    for tf, limit in [(TF_FAST, LOOKBACK_FAST), (TF_SLOW, LOOKBACK_SLOW)]:
        try:
            df = fetch_ohlcv(sym, timeframe=tf, limit=limit, exchange_id=exchange_id)
            if len(df) < MIN_BARS.get(tf, 30):
                continue   # keep default 50 — not enough data
            c = df["close"]
            e20  = c.ewm(span=20, adjust=False).mean()
            e50  = c.ewm(span=50, adjust=False).mean()
            r    = rsi(c, 14)
            last_c   = float(c.iloc[-1])
            last_r   = float(r.iloc[-1])
            last_e20 = float(e20.iloc[-1])
            last_e50 = float(e50.iloc[-1])
            score = 50.0
            score += 15 if last_c > last_e20 else -15
            score += 15 if last_c > last_e50 else -15
            score += 10 if last_r > 50 else -10
            fast_e = c.ewm(span=12, adjust=False).mean()
            slow_e = c.ewm(span=26, adjust=False).mean()
            macd_h = float((fast_e - slow_e).iloc[-1])
            score += 10 if macd_h > 0 else -10
            # Floor at 10 (not 1): avoids treating a fully-bearish 1h as
            # 'confirmed bearish signal' when it's just a short pullback
            clipped = float(np.clip(score, 10, 100))
            signals[tf] = clipped
        except Exception:
            pass   # keep default 50
    return signals


print("✅ Fetch functions defined (OHLCV, F&G, Funding, Dominance, News, MTF)")

## 4) Technical Indicators

All indicators are implemented from scratch to avoid dependency issues.

In [ ]:
# ── Core helpers ────────────────────────────────────────────────────

def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()


def rsi(close: pd.Series, period: int = RSI_PERIOD) -> pd.Series:
    delta    = close.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).replace([np.inf, -np.inf], np.nan).bfill()


def macd(close: pd.Series, fast=MACD_FAST, slow=MACD_SLOW, signal=MACD_SIGNAL):
    """Returns (macd_line, signal_line, histogram)."""
    fast_ema = ema(close, fast)
    slow_ema = ema(close, slow)
    macd_line    = fast_ema - slow_ema
    signal_line  = ema(macd_line, signal)
    histogram    = macd_line - signal_line
    return macd_line, signal_line, histogram


def bollinger_bands(close: pd.Series, period=BB_PERIOD, num_std=BB_STD):
    """Returns (upper, middle, lower, %B, bandwidth)."""
    middle = close.rolling(period).mean()
    std    = close.rolling(period).std()
    upper  = middle + num_std * std
    lower  = middle - num_std * std
    pct_b  = (close - lower) / (upper - lower + 1e-10)   # 0=lower band, 1=upper band
    bw     = (upper - lower) / (middle + 1e-10)           # bandwidth (squeeze proxy)
    return upper, middle, lower, pct_b, bw


def atr(high: pd.Series, low: pd.Series, close: pd.Series, period=ATR_PERIOD) -> pd.Series:
    """Average True Range."""
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()


def adx(high: pd.Series, low: pd.Series, close: pd.Series, period=ADX_PERIOD):
    """Returns (ADX, +DI, -DI)."""
    prev_high  = high.shift(1)
    prev_low   = low.shift(1)
    prev_close = close.shift(1)

    up   = high - prev_high
    down = prev_low - low

    plus_dm  = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)

    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)

    atr_s    = tr.ewm(span=period, adjust=False).mean()
    plus_di  = 100 * pd.Series(plus_dm,  index=close.index).ewm(span=period, adjust=False).mean() / atr_s
    minus_di = 100 * pd.Series(minus_dm, index=close.index).ewm(span=period, adjust=False).mean() / atr_s

    dx  = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-10)
    adx_val = dx.ewm(span=period, adjust=False).mean()
    return adx_val, plus_di, minus_di


def stochastic(high: pd.Series, low: pd.Series, close: pd.Series,
               k_period=STOCH_K, d_period=STOCH_D):
    """Returns (%K, %D)."""
    lowest  = low.rolling(k_period).min()
    highest = high.rolling(k_period).max()
    k = 100 * (close - lowest) / (highest - lowest + 1e-10)
    d = k.rolling(d_period).mean()
    return k, d


def obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """On-Balance Volume."""
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum()


def vwap_rolling(high, low, close, volume, period=20) -> pd.Series:
    """Rolling VWAP approximation."""
    typical = (high + low + close) / 3
    return (typical * volume).rolling(period).sum() / volume.rolling(period).sum()


def percentile_rank(series: pd.Series, lookback: int = 100) -> pd.Series:
    """Rolling percentile rank (0–100) of current value over lookback window."""
    return series.rolling(lookback).apply(
        lambda x: stats.percentileofscore(x[:-1], x[-1]) if len(x) > 1 else 50,
        raw=True
    )


print("✅ Indicator functions defined")

## 5) Indicator Assembly

In [ ]:
def add_all_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Compute all indicators and attach to a copy of df."""
    out = df.copy()
    c, h, l, v = out["close"], out["high"], out["low"], out["volume"]

    # EMAs
    for p in EMA_PERIODS:
        out[f"ema{p}"] = ema(c, p)

    # RSI
    out["rsi"]    = rsi(c, RSI_PERIOD)
    out["rsi_ema"] = ema(out["rsi"], 9)   # smoothed RSI for divergence

    # MACD
    out["macd"], out["macd_signal"], out["macd_hist"] = macd(c)
    out["macd_hist_prev"] = out["macd_hist"].shift(1)
    out["macd_expanding"] = out["macd_hist"] > out["macd_hist_prev"]  # histogram growing

    # Bollinger Bands
    out["bb_upper"], out["bb_mid"], out["bb_lower"], out["bb_pct"], out["bb_bw"] = bollinger_bands(c)
    out["bb_bw_pct"] = percentile_rank(out["bb_bw"], 120)   # squeeze = low percentile

    # ATR & ATR %
    out["atr"]     = atr(h, l, c, ATR_PERIOD)
    out["atr_pct"] = out["atr"] / c * 100   # normalized
    out["atr_pct_rank"] = percentile_rank(out["atr_pct"], 120)

    # ADX & DI
    out["adx"], out["plus_di"], out["minus_di"] = adx(h, l, c)

    # Stochastic
    out["stoch_k"], out["stoch_d"] = stochastic(h, l, c)

    # OBV & OBV EMA
    out["obv"]     = obv(c, v)
    out["obv_ema"] = ema(out["obv"], OBV_EMA_SPAN)
    out["obv_trend"] = out["obv"] > out["obv_ema"]   # OBV above its EMA → accumulation

    # Rolling VWAP
    out["vwap"] = vwap_rolling(h, l, c, v, 20)

    # Volume metrics
    out["vol_sma20"] = v.rolling(20).mean()
    out["vol_ratio"] = v / out["vol_sma20"]   # > 1.5 = significant volume
    out["vol_trend"] = ema(v, 10) > ema(v, 30)  # short-term vol trend

    # Returns
    out["ret_1"]  = c.pct_change(1)
    out["ret_6"]  = c.pct_change(6)    # ~1 day on 4h
    out["ret_18"] = c.pct_change(18)   # ~3 days
    out["ret_42"] = c.pct_change(42)   # ~7 days

    # Price vs EMAs (normalized distance)
    for p in EMA_PERIODS:
        out[f"dist_ema{p}"] = (c - out[f"ema{p}"]) / out[f"ema{p}"] * 100

    # EMA slopes (6-bar)
    for p in [9, 20, 50]:
        out[f"slope_ema{p}"] = out[f"ema{p}"].diff(6)

    return out


print("✅ add_all_indicators() defined")

## 6) Multi-Layer Scoring Engine

Each sub-scorer returns a value in **[0, 100]** where:
- **> 60** = bullish contribution
- **40–60** = neutral
- **< 40** = bearish contribution

In [ ]:
def _safe_float(v, default=50.0) -> float:
    try:
        f = float(v)
        return f if np.isfinite(f) else default
    except Exception:
        return default


# ── 6a. Momentum Score (RSI + MACD + Stochastic) ─────────────────────

def score_momentum(last, prev, df) -> float:
    score = 50.0

    # RSI (40% of sub-score)
    r = _safe_float(last.get("rsi", 50))
    r_prev = _safe_float(prev.get("rsi", 50))
    rsi_score = np.clip((r - 30) / 40 * 100, 0, 100)   # 30→0, 70→100
    rsi_momentum = 10 if r > r_prev else -10             # rising RSI bonus
    rsi_contrib = np.clip(rsi_score + rsi_momentum, 0, 100)

    # Stochastic (30% of sub-score)
    k = _safe_float(last.get("stoch_k", 50))
    d = _safe_float(last.get("stoch_d", 50))
    stoch_score = np.clip((k - 20) / 60 * 100, 0, 100)  # 20→0, 80→100
    stoch_cross = 10 if k > d else -10
    stoch_contrib = np.clip(stoch_score + stoch_cross, 0, 100)

    # MACD (30% of sub-score)
    macd_val  = _safe_float(last.get("macd", 0))
    macd_sig  = _safe_float(last.get("macd_signal", 0))
    macd_hist = _safe_float(last.get("macd_hist", 0))
    expanding = bool(last.get("macd_expanding", False))

    macd_score = 50.0
    if macd_val > macd_sig:
        macd_score += 20
    else:
        macd_score -= 20
    if macd_hist > 0:
        macd_score += 15
    if expanding:
        macd_score += 10
    macd_contrib = np.clip(macd_score, 0, 100)

    return float(0.40 * rsi_contrib + 0.30 * stoch_contrib + 0.30 * macd_contrib)


# ── 6b. Trend Score (EMA alignment + ADX + price position) ───────────

def score_trend(last, df) -> float:
    score = 50.0
    c = _safe_float(last.get("close", 0))

    # EMA alignment: score by how many EMAs price is above
    ema_vals = [_safe_float(last.get(f"ema{p}", 0)) for p in EMA_PERIODS]
    above_count = sum(1 for e in ema_vals if c > e)
    # Weighted: being above ema200 is worth more
    ema_weights = [1, 1.5, 2, 2, 3]  # 9, 20, 50, 100, 200
    weighted_above = sum(w for w, e in zip(ema_weights, ema_vals) if c > e)
    max_weight = sum(ema_weights)
    ema_score = (weighted_above / max_weight) * 100

    # EMA slope alignment
    slopes_up = sum(1 for p in [9, 20, 50]
                    if _safe_float(last.get(f"slope_ema{p}", 0)) > 0)
    slope_score = slopes_up / 3 * 100

    # ADX: trend strength (not direction)
    adx_val  = _safe_float(last.get("adx", 25))
    plus_di  = _safe_float(last.get("plus_di", 25))
    minus_di = _safe_float(last.get("minus_di", 25))

    if adx_val > 25:   # meaningful trend exists
        adx_direction = 70 if plus_di > minus_di else 30
    else:
        adx_direction = 50  # choppy, no trend

    # Price vs VWAP
    vwap_val = _safe_float(last.get("vwap", c))
    vwap_score = 65 if c > vwap_val else 35

    return float(0.35 * ema_score + 0.25 * slope_score + 0.25 * adx_direction + 0.15 * vwap_score)


# ── 6c. Volume Score (OBV trend + volume expansion + buy pressure) ────

def score_volume(last, df) -> float:
    # OBV trend
    obv_above = bool(last.get("obv_trend", False))
    obv_score = 65 if obv_above else 35

    # Volume ratio vs 20-bar avg
    vol_ratio = _safe_float(last.get("vol_ratio", 1.0))
    close_val = _safe_float(last.get("close", 0))
    open_val  = _safe_float(last.get("open", close_val))

    # High volume + up candle = buying pressure
    if vol_ratio > 1.5:
        if close_val > open_val:
            vol_pressure = 80
        else:
            vol_pressure = 20   # high volume sell = bearish
    elif vol_ratio > 1.0:
        vol_pressure = 55 if close_val > open_val else 45
    else:
        vol_pressure = 50   # low volume = neutral

    # Volume trend (short > long EMA)
    vol_trend = bool(last.get("vol_trend", False))
    vol_trend_score = 65 if vol_trend else 35

    raw = float(0.40 * obv_score + 0.40 * vol_pressure + 0.20 * vol_trend_score)

    # ── Low-volume bullish penalty (NEW) ─────────────────────────────
    # If volume is well below average on a bullish signal, it lacks conviction.
    # This directly addresses the April 6 false 85/100 where vol ratio was 0.2-0.6x.
    if vol_ratio < LOW_VOL_THRESHOLD and raw > 55:
        # Scale penalty: the lower the volume, the bigger the pull toward neutral
        penalty_strength = (LOW_VOL_THRESHOLD - vol_ratio) / LOW_VOL_THRESHOLD
        raw = raw - (raw - 50) * penalty_strength * 0.5

    return float(np.clip(raw, 0, 100))


# ── 6d. Volatility Regime Score ───────────────────────────────────────

def score_volatility(last, df) -> float:
    """
    v3.5 fix: BB squeeze (low bandwidth) previously scored as bearish because
    bb_pct < 0.5 returned a low score. But a squeeze is NOT bearish — it means
    price is coiling and a breakout is pending in *either* direction.
    The scorer now treats squeezes as neutral (50) unless price has already
    broken above or below the midline *out of* the squeeze, which is directional.

    Scoring logic:
    - bb_pct 0.0-1.0: where price sits in the bands (0=lower, 0.5=mid, 1=upper)
    - Active squeeze (bb_bw_pct < 20): return 50 ± small directional nudge only
    - Post-squeeze (bb_bw_pct 20-40): bb_pct matters more — direction emerging
    - Normal (bb_bw_pct > 40): full bb_pct scoring
    - High ATR (> 85th pct): pull toward 50 (extreme vol = uncertainty)
    """
    bb_pct    = _safe_float(last.get("bb_pct", 0.5))
    bb_bw_pct = _safe_float(last.get("bb_bw_pct", 50))
    atr_rank  = _safe_float(last.get("atr_pct_rank", 50))

    if bb_bw_pct < 20:
        # Active squeeze — coiling, direction unclear
        # Only small nudge based on where price is in the bands
        nudge = (bb_pct - 0.5) * 20   # max ±10 points
        vol_score = 50 + nudge
    elif bb_bw_pct < 40:
        # Post-squeeze / emerging direction — moderate weighting
        bb_score  = np.clip(bb_pct * 100, 0, 100)
        vol_score = 50 + (bb_score - 50) * 0.5   # half weight
    else:
        # Normal regime — full bb_pct scoring
        vol_score = np.clip(bb_pct * 100, 0, 100)

    # ATR extreme: very high volatility = pull toward 50 (uncertainty)
    if atr_rank > 85:
        vol_score = vol_score + (50 - vol_score) * 0.3

    return float(np.clip(vol_score, 0, 100))


# ── 6g. Funding Rate Score (NEW) ─────────────────────────────────────

def score_funding(coin: str, funding_map: dict) -> float:
    """
    Use pre-fetched funding rate data to score market leverage bias.
    Returns the pre-computed score from fetch_funding_rates().
    """
    data = funding_map.get(coin, {})
    return float(data.get("score", 50.0))


# ── 6h. Multi-Timeframe Confirmation Score (NEW) ──────────────────────

def score_mtf(coin: str, primary_score: float, mtf_signals: dict) -> float:
    """
    Compares the primary 4h signal against 1h and 1d signals.
    - All three timeframes agree → amplify signal (pull toward primary)
    - Two agree → neutral amplification
    - Only 4h bullish but 1h+1d bearish → dampen signal (pull toward 50)

    This directly prevents single-timeframe false signals like April 6.
    """
    fast_score = mtf_signals.get(TF_FAST, 50.0)
    slow_score = mtf_signals.get(TF_SLOW, 50.0)

    # Determine direction agreement
    primary_bull = primary_score >= 55
    primary_bear = primary_score <= 45
    fast_bull    = fast_score >= 55
    fast_bear    = fast_score <= 45
    slow_bull    = slow_score >= 55
    slow_bear    = slow_score <= 45

    if primary_bull:
        agreements = sum([fast_bull, slow_bull])
    elif primary_bear:
        agreements = sum([fast_bear, slow_bear])
    else:
        agreements = 1  # neutral primary = neutral MTF score

    if agreements == 2:
        # Full confluence — strong confirmation, return boosted primary direction
        mtf_score = np.clip(primary_score + (primary_score - 50) * 0.2, 0, 100)
    elif agreements == 1:
        # Partial — return average of all three timeframes
        mtf_score = (primary_score + fast_score + slow_score) / 3
    else:
        # No agreement — primary is isolated, dampen toward 50
        mtf_score = 50 + (primary_score - 50) * 0.25

    return float(np.clip(mtf_score, 0, 100))


# ── 6e. Cross-Asset Correlation Score ────────────────────────────────

def score_correlation(df: pd.DataFrame, btc_df: pd.DataFrame) -> float:
    """
    Score based on:
    - BTC trend: if BTC is bullish, alts tend to follow
    - Alt beta: high-beta alts amplify BTC moves
    - Rolling correlation: high correlation = dependent on BTC sentiment

    FIX: Previously returned 0 when corr_val was NaN (silent _safe_float default).
    Now uses 0.7 as the assumed correlation (crypto assets are highly correlated)
    and always returns BTC-driven score when BTC data is available.
    """
    if btc_df is None or df is None:
        return 50.0

    try:
        # BTC trend signal (always computed — does not depend on index alignment)
        btc_last   = btc_df.iloc[-1]
        btc_close  = _safe_float(btc_last.get("close", 0))
        btc_ema50  = _safe_float(btc_last.get("ema50",  btc_close))
        btc_ema200 = _safe_float(btc_last.get("ema200", btc_close))
        btc_rsi    = _safe_float(btc_last.get("rsi", 50))
        btc_adx    = _safe_float(btc_last.get("adx", 20))
        btc_pdi    = _safe_float(btc_last.get("plus_di", 25))
        btc_mdi    = _safe_float(btc_last.get("minus_di", 25))

        btc_bullish = (
            int(btc_close > btc_ema50) +
            int(btc_close > btc_ema200) +
            int(btc_rsi > 50) +
            int(btc_adx > 25 and btc_pdi > btc_mdi)
        )
        btc_score = btc_bullish / 4 * 100   # 0, 25, 50, 75, or 100

        # Attempt rolling correlation — use 0.7 as fallback (crypto is highly correlated)
        corr_val = 0.7  # default: assume moderate-high correlation
        try:
            # Use reset_index to avoid timezone/freq mismatch issues
            alt_ret = df["ret_1"].dropna().tail(60).reset_index(drop=True)
            btc_ret = btc_df["ret_1"].dropna().tail(60).reset_index(drop=True)
            min_len = min(len(alt_ret), len(btc_ret))
            if min_len >= 30:
                alt_ret = alt_ret.iloc[:min_len]
                btc_ret = btc_ret.iloc[:min_len]
                computed_corr = alt_ret.rolling(20).corr(btc_ret).iloc[-1]
                if np.isfinite(computed_corr):
                    corr_val = float(computed_corr)
        except Exception:
            pass  # keep default corr_val of 0.7

        # Beta adjustment
        try:
            alt_r = df["ret_1"].dropna().tail(60).reset_index(drop=True)
            btc_r = btc_df["ret_1"].dropna().tail(60).reset_index(drop=True)
            min_len = min(len(alt_r), len(btc_r))
            alt_r = alt_r.iloc[:min_len]
            btc_r = btc_r.iloc[:min_len]
            if btc_r.std() > 0 and alt_r.std() > 0:
                beta = np.cov(alt_r, btc_r)[0, 1] / np.var(btc_r)
                beta = float(np.clip(beta, 0.5, 3.0))
            else:
                beta = 1.0
        except Exception:
            beta = 1.0

        # Blend: high correlation = follow BTC score closely
        # Low correlation = partial independence, drift toward 50
        corr_weight = np.clip(corr_val, 0, 1)
        blended = corr_weight * btc_score + (1 - corr_weight) * 50

        # High-beta alts amplify BTC direction (pull further from 50)
        if beta > 1.2:
            amplify = (blended - 50) * min(beta * 0.15, 0.4)
            blended = blended + amplify

        return float(np.clip(blended, 0, 100))

    except Exception:
        return 50.0


# ── 6f. Sentiment Score (Fear & Greed) ───────────────────────────────

def score_sentiment(fg: dict, coin: str) -> float:
    """
    Fear & Greed is a contrarian indicator at extremes.

    Updated mapping with stronger weighting at extremes (< 10 or > 90)
    because historical data shows these levels reliably precede reversals:
    - Extreme Fear (< 10) → strong contrarian buy (score 90)
    - Extreme Fear (10-20) → contrarian buy (score 78)
    - Fear (20-35) → cautiously bullish (score 65)
    - Fear (35-45) → mild bullish lean (score 57)
    - Neutral (45-55) → neutral (score 50)
    - Greed (55-70) → mild bearish lean (score 43)
    - Greed (70-85) → cautiously bearish (score 33)
    - Extreme Greed (> 85) → strong contrarian sell (score 18)

    Non-BTC coins apply BTC sentiment with slight damping.
    """
    v     = fg.get("value", 50)
    trend = fg.get("trend", "flat")
    avg7  = fg.get("avg7", v)

    # Contrarian mapping — stronger at extremes
    if v < 10:
        base = 90   # historically rare extreme — strong contrarian buy
    elif v < 20:
        base = 78
    elif v < 35:
        base = 65
    elif v < 45:
        base = 57
    elif v < 55:
        base = 50
    elif v < 70:
        base = 43
    elif v < 85:
        base = 33
    else:
        base = 18   # extreme greed = strong contrarian sell

    # Trend adjustment: a rising F&G from extreme fear is MORE bullish
    # (momentum confirming the contrarian signal)
    if trend == "rising" and v < 30:
        trend_adj = 8   # recovering from extreme fear = stronger buy
    elif trend == "rising" and v < 50:
        trend_adj = 4
    elif trend == "falling" and v > 70:
        trend_adj = -8  # deteriorating from extreme greed = stronger sell
    elif trend == "falling" and v > 50:
        trend_adj = -4
    else:
        trend_adj = 0

    # 7-day divergence: current vs avg — adds context
    divergence_adj = np.clip((v - avg7) * 0.1, -5, 5)

    # Damp for non-BTC (alts are more idiosyncratic)
    damp = 0.75 if coin != "BTC" else 1.0
    raw = 50 + (base - 50 + trend_adj) * damp + divergence_adj
    return float(np.clip(raw, 0, 100))


print("✅ All 8 scoring functions defined (incl. funding + MTF)")

## 7) Composite Scorer & Signal Classifier

In [ ]:
def composite_score(df: pd.DataFrame, btc_df: pd.DataFrame,
                    fg: dict, coin: str,
                    funding_map: dict = None,
                    mtf_signals: dict = None,
                    dominance: dict = None,
                    news_scores: dict = None,
                    weights: dict = WEIGHTS) -> dict:
    """
    Returns a dict with all sub-scores and the weighted composite.
    Now includes funding rates, multi-timeframe confirmation,
    BTC dominance dampening, and news sentiment blending.
    """
    last = df.iloc[-1].to_dict()
    prev = df.iloc[-2].to_dict()

    s_momentum    = score_momentum(last, prev, df)
    s_trend       = score_trend(last, df)
    s_volume      = score_volume(last, df)
    s_volatility  = score_volatility(last, df)
    s_correlation = score_correlation(df, btc_df)

    # ── Sentiment: blend F&G with news if available ──────────────────
    s_fg = score_sentiment(fg, coin)
    if news_scores and coin in news_scores and news_scores[coin] != 50.0:
        news_s = float(news_scores[coin])
        s_sentiment = 0.6 * s_fg + 0.4 * news_s
    else:
        s_sentiment = s_fg

    # ── Funding rate score ───────────────────────────────────────────
    s_funding = score_funding(coin, funding_map or {})

    # ── Primary composite (without MTF) ─────────────────────────────
    # Used as input to MTF scorer
    primary_w_sum = (weights["momentum"] + weights["trend"] + weights["volume"] +
                     weights["volatility"] + weights["correlation"] + weights["sentiment"] +
                     weights["funding"])
    primary_composite = (
        weights["momentum"]    * s_momentum +
        weights["trend"]       * s_trend +
        weights["volume"]      * s_volume +
        weights["volatility"]  * s_volatility +
        weights["correlation"] * s_correlation +
        weights["sentiment"]   * s_sentiment +
        weights["funding"]     * s_funding
    ) / primary_w_sum * 100

    # ── MTF score ────────────────────────────────────────────────────
    s_mtf = score_mtf(coin, primary_composite, mtf_signals or {})

    # ── Full composite ───────────────────────────────────────────────
    composite = (
        weights["momentum"]    * s_momentum +
        weights["trend"]       * s_trend +
        weights["volume"]      * s_volume +
        weights["volatility"]  * s_volatility +
        weights["correlation"] * s_correlation +
        weights["sentiment"]   * s_sentiment +
        weights["funding"]     * s_funding +
        weights["mtf"]         * s_mtf
    )

    # ── BTC dominance dampening for alts ────────────────────────────
    if dominance and coin != "BTC":
        dom_trend = dominance.get("trend", "neutral")
        if dom_trend == "rising_strong" and composite > 55:
            composite = composite - (composite - 50) * 0.25
        elif dom_trend == "rising" and composite > 60:
            composite = composite - (composite - 50) * 0.12
        elif dom_trend == "falling_strong" and composite < 45:
            composite = composite + (50 - composite) * 0.20

    # ── Hard volume cap (NEW) ────────────────────────────────────────
    # ── Hard volume cap with proportional scaling (v3.2 fix) ────────
    # v3.1 bug: hard min(composite, 64) caused ALL low-vol coins to land
    # exactly on 64, masking real differences between them.
    # Fix: scale the excess above the cap proportionally, preserving rank.
    # e.g. composites of 80, 75, 70 with cap=64 become ~64, 61, 58
    vol_ratio = _safe_float(last.get("vol_ratio", 1.0))
    if composite > 50:
        if vol_ratio < VOL_CAP_BULLISH:
            # Very low volume: scale down into Mildly Bullish range (50-65)
            excess = composite - 50
            composite = 50 + excess * 0.45   # compress excess by 55%
            composite = min(composite, 64.9)  # hard ceiling still applies
        elif vol_ratio < VOL_CAP_STRONG:
            # Low volume: scale down into Bullish range (50-72)
            excess = composite - 50
            composite = 50 + excess * 0.70   # compress excess by 30%
            composite = min(composite, 71.9)  # hard ceiling still applies
    elif composite < 50:
        if vol_ratio < VOL_CAP_BEAR:
            excess = 50 - composite
            composite = 50 - excess * 0.60   # soften strong bearish signals too
            composite = max(composite, 37.0)

    return {
        "momentum":    round(s_momentum, 1),
        "trend":       round(s_trend, 1),
        "volume":      round(s_volume, 1),
        "volatility":  round(s_volatility, 1),
        "correlation": round(s_correlation, 1),
        "sentiment":   round(s_sentiment, 1),
        "funding":     round(s_funding, 1),
        "mtf":         round(s_mtf, 1),
        "composite":   round(composite, 1),
    }


def classify_signal(scores: dict, coin: str) -> dict:
    """
    Translate composite score into a readable signal with confidence.
    Neutral band: 47-55. Scores 38-47 = Mildly Bearish.
    """
    c = scores["composite"]

    if c >= 72:
        direction = "🟢 Strong Bullish"
        confidence = "High"
    elif c >= STRONG_SIGNAL_THRESHOLD:
        direction = "🟩 Bullish"
        confidence = "Moderate"
    elif c >= 55:
        direction = "🔆 Mildly Bullish"
        confidence = "Low"
    elif c >= 47:
        direction = "⬜ Neutral"
        confidence = "Low"
    elif c >= 38:
        direction = "🔻 Mildly Bearish"
        confidence = "Low"
    elif c >= 28:
        direction = "🟥 Bearish"
        confidence = "Moderate"
    else:
        direction = "🔴 Strong Bearish"
        confidence = "High"

    # Agreement check across all 8 sub-scores
    # v3.5: MTF uses tighter bearish threshold (≤20 not ≤45) because
    # the floor of 10 represents fully-bearish 1h — treating 10 the same
    # as 45 (barely bearish) was inflating the bearish count.
    sub_score_map = {
        "momentum":    scores["momentum"],
        "trend":       scores["trend"],
        "volume":      scores["volume"],
        "volatility":  scores["volatility"],
        "correlation": scores["correlation"],
        "sentiment":   scores["sentiment"],
        "funding":     scores["funding"],
        "mtf":         scores["mtf"],
}
    BEAR_THRESH = {"mtf": 20}
    bullish_count = sum(1 for k, s in sub_score_map.items() if s >= 55)
    bearish_count = sum(1 for k, s in sub_score_map.items()
                        if s <= BEAR_THRESH.get(k, 45))
    agreement = f"{bullish_count}/8 bullish, {bearish_count}/8 bearish"

    # Key driver
    sub_map = {
        "momentum":    scores["momentum"],
        "trend":       scores["trend"],
        "volume":      scores["volume"],
        "volatility":  scores["volatility"],
        "correlation": scores["correlation"],
        "sentiment":   scores["sentiment"],
        "funding":     scores["funding"],
        "mtf":         scores["mtf"],
    }
    strongest_bull = max(sub_map, key=lambda k: sub_map[k])
    strongest_bear = min(sub_map, key=lambda k: sub_map[k])

    if c >= 50:
        key_driver = f"+{strongest_bull} ({sub_map[strongest_bull]:.0f})"
    else:
        key_driver = f"-{strongest_bear} ({sub_map[strongest_bear]:.0f})"

    # Flag funding conflict: bullish signal but funding is bearish
    funding_s = scores.get("funding", 50)
    if c >= 55 and funding_s <= 40:
        key_driver += " ⚠️funding"

    # ── Divergence detection (v3.4) ──────────────────────────────────
    # Threshold raised 70→80pt: at 70 almost every coin with bearish 1h
    # and bullish sentiment triggered HIGH, diluting the flag's meaning.
    # Also detect when 1h is the sole bearish driver (may be a short pullback)
    # and note it rather than treating it as a confirmed conflict.
    sub_vals = list(sub_map.values())
    sub_range = max(sub_vals) - min(sub_vals)
    bull_subs = [k for k, v in sub_map.items() if v >= 60]
    bear_subs = [k for k, v in sub_map.items() if v <= 40]

    # Check if MTF (1h) is the sole or dominant bearish driver
    mtf_score = sub_map.get("mtf", 50)
    non_mtf_bear = [k for k, v in sub_map.items() if v <= 40 and k != "mtf"]
    mtf_sole_bear = mtf_score <= 30 and len(non_mtf_bear) == 0

    if sub_range >= 80 and len(bull_subs) >= 2 and len(bear_subs) >= 2:
        note = " (1h pullback only)" if mtf_sole_bear else ""
        divergence = f"⚡ HIGH ({sub_range:.0f}pt) — {','.join(bull_subs)} vs {','.join(bear_subs)}{note}"
        confidence = "Low"
    elif sub_range >= 60 and len(bull_subs) >= 1 and len(bear_subs) >= 1:
        note = " — 1h weak, watch for recovery" if mtf_sole_bear else ""
        divergence = f"↕ MOD ({sub_range:.0f}pt){note}"
    else:
        divergence = "—"

    return {
        "direction":   direction,
        "confidence":  confidence,
        "agreement":   agreement,
        "key_driver":  key_driver,
        "divergence":  divergence,
    }


print("✅ Composite scorer and classifier defined")

## 8) Bounce Quality Detector

Analyses whether a bullish signal is a **sustainable move** or a **short-lived bounce** that is likely to reverse.

Based on observed patterns from April 2026 (ceasefire-driven headline bounce) and historical crypto bounce behaviour:

| Pattern | Dead-Cat Bounce | Sustainable Move |
|---|---|---|
| Volume | Low (< 0.7x avg) across multiple bars | High + sustained (> 1.2x for 3+ bars) |
| Prior trend | Strongly bearish (below EMA50 + EMA200) | Neutral or recovering |
| Speed of rise | Sharp spike (> 5% in 1-2 bars) | Gradual accumulation |
| RSI behaviour | Jumped from oversold but still < 55 | Steady climb above 55 |
| Daily trend | Bearish (1d score < 45) | Neutral or bullish |
| F&G context | Still Extreme Fear despite spike | Recovering from Fear |
| EMA structure | Price below EMA200 and EMA50 still falling | Price reclaimed EMA50, EMA200 flat/rising |
| MACD daily | Still negative | Crossed above signal |


In [ ]:
def detect_bounce_quality(coin: str, df: pd.DataFrame,
                          mtf_signals: dict, fg: dict) -> dict:
    """
    Analyses whether the current bullish move is a sustainable breakout
    or a short-lived bounce likely to reverse.

    Returns a dict with:
    - quality: 'Sustainable' | 'Mixed' | 'Likely Dead-Cat'
    - confidence: 'High' | 'Moderate' | 'Low'
    - warning: plain-English warning string (empty if sustainable)
    - flags: list of specific red flags detected
    - score: 0-100 (0 = certain dead-cat, 100 = certain sustainable)

    Only meaningful when the composite signal is Bullish or Strong Bullish.
    """
    if df is None or len(df) < 30:
        return {"quality": "Unknown", "confidence": "Low",
                "warning": "", "flags": [], "score": 50}

    last  = df.iloc[-1]
    flags = []          # red flags = bounce likely to fail
    green = []          # green flags = move likely to sustain
    score = 50.0        # start neutral

    close   = _safe_float(last.get("close", 0))
    ema50   = _safe_float(last.get("ema50", close))
    ema200  = _safe_float(last.get("ema200", close))
    ema20   = _safe_float(last.get("ema20", close))
    rsi_now = _safe_float(last.get("rsi", 50))
    adx     = _safe_float(last.get("adx", 20))
    plus_di = _safe_float(last.get("plus_di", 25))
    minus_di= _safe_float(last.get("minus_di", 25))

    # ── 1. Volume quality over last 6 bars ───────────────────────────
    recent_vols = df["vol_ratio"].tail(6).dropna()
    avg_recent_vol = float(recent_vols.mean()) if len(recent_vols) > 0 else 1.0
    high_vol_bars  = int((recent_vols > 1.2).sum())

    if avg_recent_vol < 0.6:
        flags.append(f"Thin volume: avg {avg_recent_vol:.2f}x over last 6 bars — no institutional conviction")
        score -= 20
    elif avg_recent_vol < 0.8:
        flags.append(f"Below-average volume: {avg_recent_vol:.2f}x — move lacks broad participation")
        score -= 10
    elif high_vol_bars >= 3:
        green.append(f"Sustained volume: {high_vol_bars}/6 bars above 1.2x avg")
        score += 15

    # ── 2. Speed of the move (sharp spike = bounce signature) ────────
    ret_2bar = _safe_float(last.get("ret_6", 0)) * 100   # ~1 day on 4h
    ret_6bar = _safe_float(last.get("ret_18", 0)) * 100  # ~3 days

    if ret_2bar > 6 and avg_recent_vol < 0.8:
        flags.append(f"Sharp spike: +{ret_2bar:.1f}% in ~1 day on thin volume — classic dead-cat pattern")
        score -= 25
    elif ret_2bar > 4 and avg_recent_vol < 1.0:
        flags.append(f"Fast move (+{ret_2bar:.1f}% in ~1 day) without volume confirmation")
        score -= 12
    elif ret_6bar > 0 and ret_6bar < 8 and avg_recent_vol >= 1.0:
        green.append(f"Measured 3-day gain of +{ret_6bar:.1f}% with volume support — controlled move")
        score += 10

    # ── 3. EMA structure — is the underlying trend actually broken? ───
    above_ema50  = close > ema50
    above_ema200 = close > ema200
    ema50_slope  = _safe_float(last.get("slope_ema50", 0))
    ema20_slope  = _safe_float(last.get("slope_ema20", 0))

    if not above_ema200 and not above_ema50:
        flags.append("Price still below both EMA50 and EMA200 — bounce within a downtrend, not a reversal")
        score -= 20
    elif not above_ema200 and above_ema50:
        flags.append("Price above EMA50 but still below EMA200 — intermediate bounce, long-term trend bearish")
        score -= 8
    elif above_ema200 and above_ema50:
        green.append("Price above both EMA50 and EMA200 — trend structure is bullish")
        score += 15

    if ema50_slope < 0:
        flags.append("EMA50 still declining — trend has not reversed, price is fighting the slope")
        score -= 10
    elif ema50_slope > 0 and ema20_slope > 0:
        green.append("EMA20 and EMA50 both turning up — trend beginning to reverse")
        score += 12

    # ── 4. RSI behaviour — did it recover or just spike? ─────────────
    rsi_vals = df["rsi"].tail(10).dropna()
    rsi_min_recent = float(rsi_vals.min()) if len(rsi_vals) > 0 else 50
    rsi_trajectory = float(rsi_vals.iloc[-1] - rsi_vals.iloc[0]) if len(rsi_vals) > 1 else 0

    if rsi_min_recent < 30 and rsi_now < 55:
        flags.append(f"RSI bounced from oversold ({rsi_min_recent:.0f}) but only reached {rsi_now:.0f} — typical relief bounce, not momentum shift")
        score -= 15
    elif rsi_now > 60 and rsi_trajectory > 15:
        green.append(f"RSI climbed steadily to {rsi_now:.0f} (+{rsi_trajectory:.0f} over 10 bars) — genuine momentum")
        score += 12
    elif rsi_now < 50:
        flags.append(f"RSI at {rsi_now:.0f} — still below 50, momentum has not confirmed the bullish signal")
        score -= 8

    # ── 5. Daily timeframe confirmation ──────────────────────────────
    daily_score = mtf_signals.get(TF_SLOW, 50.0)
    if daily_score < 40:
        flags.append(f"Daily chart bearish ({daily_score:.0f}/100) — higher timeframe trend is against this move")
        score -= 18
    elif daily_score < 50:
        flags.append(f"Daily chart weak ({daily_score:.0f}/100) — no confirmation on higher timeframe")
        score -= 8
    elif daily_score > 60:
        green.append(f"Daily chart supportive ({daily_score:.0f}/100) — higher timeframe agrees")
        score += 15

    # ── 6. Fear & Greed context ──────────────────────────────────────
    fg_val   = fg.get("value", 50)
    fg_trend = fg.get("trend", "flat")

    if fg_val < 20 and fg_trend == "flat":
        flags.append(f"F&G still at {fg_val} (Extreme Fear) despite price spike — sentiment has not recovered, suggesting headline-driven move")
        score -= 15
    elif fg_val < 20 and fg_trend == "rising":
        flags.append(f"F&G at {fg_val} but rising — possible early recovery, watch for continuation")
        score -= 5
    elif fg_val > 35 and fg_trend == "rising":
        green.append(f"F&G recovering ({fg_val}, trend: rising) — sentiment backing the move")
        score += 10

    # ── 7. Historical bounce pattern match ───────────────────────────
    # Check how many times in the lookback period a similar spike
    # (sharp move up after sustained downtrend) resulted in continuation vs reversal
    try:
        close_s = df["close"]
        # Find historical instances of 3%+ single-bar moves from below EMA50
        prior_below_ema50 = df["close"] < df["ema50"]
        big_move = df["ret_1"].abs() > 0.03
        spike_bars = df[prior_below_ema50 & big_move].index

        reversals = 0
        continuations = 0
        for bar in spike_bars[:-1]:   # exclude current bar
            idx = df.index.get_loc(bar)
            if idx + 6 >= len(df):
                continue
            spike_ret   = float(df["ret_1"].iloc[idx])
            forward_ret = float(df["close"].iloc[idx + 6] / df["close"].iloc[idx] - 1)
            if spike_ret > 0:   # upward spike from below EMA50
                if forward_ret < -0.02:   # reversed within ~1 day
                    reversals += 1
                elif forward_ret > 0.02:
                    continuations += 1

        total = reversals + continuations
        if total >= 5:
            reversal_rate = reversals / total * 100
            if reversal_rate > 65:
                flags.append(f"Historical pattern: {reversal_rate:.0f}% of similar spikes from below EMA50 reversed within 24h on this coin")
                score -= 18
            elif reversal_rate > 50:
                flags.append(f"Historical pattern: {reversal_rate:.0f}% reversal rate on similar spikes — slight lean toward fade")
                score -= 8
            elif reversal_rate < 35:
                green.append(f"Historical pattern: only {reversal_rate:.0f}% of similar spikes reversed — this coin tends to follow through")
                score += 10
    except Exception:
        pass

    # ── Final classification ─────────────────────────────────────────
    score = float(np.clip(score, 0, 100))

    if score >= 65:
        quality    = "✅ Sustainable Move"
        confidence = "Moderate" if score < 80 else "High"
        warning    = ""
    elif score >= 45:
        quality    = "⚠️ Mixed — Monitor Closely"
        confidence = "Low"
        warning    = "Move shows both bullish and bearish characteristics. Wait for volume confirmation and daily chart alignment before acting."
    elif score >= 25:
        quality    = "🔴 Likely Dead-Cat Bounce"
        confidence = "Moderate"
        warning    = ("Based on historical patterns, this type of move — sharp rally on thin volume, "
                      "with price still below key EMAs and the daily trend bearish — has typically "
                      "reversed within 24-72 hours. The underlying trend is still bearish. "
                      "Consider waiting for the drop to re-establish or for a genuine breakout with volume.")
    else:
        quality    = "🔴 Strong Dead-Cat Signal"
        confidence = "High"
        warning    = ("Multiple historical dead-cat bounce signatures detected. Sharp spike on thin volume "
                      "from a bearish trend with no daily confirmation is the classic profile of a "
                      "headline-driven short-lived rally. History strongly suggests a drop follows. "
                      "Do not chase this move.")

    return {
        "quality":    quality,
        "confidence": confidence,
        "warning":    warning,
        "flags":      flags,
        "green":      green,
        "score":      round(score, 1),
    }


def run_bounce_analysis(data_map: dict, score_details: dict,
                        mtf_map: dict, fg: dict) -> dict:
    """
    Runs bounce quality detection on all coins that are currently
    showing a Bullish or Strong Bullish signal.
    Returns {coin: bounce_result}
    """
    results = {}
    for coin, scores in score_details.items():
        if scores.get("composite", 50) >= 55:   # only analyse bullish signals
            df  = data_map.get(coin)
            mtf = mtf_map.get(coin, {})
            results[coin] = detect_bounce_quality(coin, df, mtf, fg)
    return results


print("✅ Bounce Quality Detector defined")

In [ ]:
def detect_market_regime(data_map: dict, fg: dict) -> dict:
    """
    Returns a regime dict with:
    - regime: string label
    - btc_trend: Bullish/Bearish/Neutral
    - alt_breadth: % of alts in uptrend
    - volatility: Low/Normal/High/Extreme
    - fg_label + fg_value
    - summary: 2-3 sentence interpretation
    """
    btc_df = data_map.get("BTC")

    # ── BTC Trend ──
    if btc_df is not None:
        btc_last = btc_df.iloc[-1]
        btc_c    = _safe_float(btc_last.get("close", 0))
        btc_ema50  = _safe_float(btc_last.get("ema50", btc_c))
        btc_ema200 = _safe_float(btc_last.get("ema200", btc_c))
        btc_rsi    = _safe_float(btc_last.get("rsi", 50))
        btc_adx    = _safe_float(btc_last.get("adx", 20))
        btc_pdi    = _safe_float(btc_last.get("plus_di", 25))
        btc_mdi    = _safe_float(btc_last.get("minus_di", 25))
        btc_ret7   = _safe_float(btc_last.get("ret_42", 0)) * 100  # 7-day %

        btc_signals = [
            btc_c > btc_ema50,
            btc_c > btc_ema200,
            btc_rsi > 50,
            btc_adx > 25 and btc_pdi > btc_mdi,
            btc_ret7 > 0,
        ]
        btc_score = sum(btc_signals)
        if btc_score >= 4:
            btc_trend = "Bullish"
        elif btc_score >= 2:
            btc_trend = "Neutral"
        else:
            btc_trend = "Bearish"

        # Volatility from ATR %
        atr_rank = _safe_float(btc_last.get("atr_pct_rank", 50))
        if atr_rank > 85:
            vol_regime = "Extreme"
        elif atr_rank > 65:
            vol_regime = "High"
        elif atr_rank > 35:
            vol_regime = "Normal"
        else:
            vol_regime = "Low (Possible Squeeze)"
    else:
        btc_trend  = "Unknown"
        vol_regime = "Unknown"
        btc_ret7   = 0

    # ── Alt Breadth ──
    alts = [k for k in data_map if k != "BTC"]
    breadth_scores = []
    for a in alts:
        d = data_map[a]
        if d is None or len(d) < 2:
            continue
        row = d.iloc[-1]
        c   = _safe_float(row.get("close", 0))
        e20 = _safe_float(row.get("ema20", c))
        e50 = _safe_float(row.get("ema50", c))
        r   = _safe_float(row.get("rsi", 50))
        breadth_scores.append(int(c > e20) + int(c > e50) + int(r > 50))

    if breadth_scores:
        avg_breadth = np.mean(breadth_scores)
        alt_breadth_pct = avg_breadth / 3 * 100
    else:
        alt_breadth_pct = 50

    if alt_breadth_pct >= 70:
        breadth_label = "Strong (Risk-On)"
    elif alt_breadth_pct >= 50:
        breadth_label = "Moderate"
    elif alt_breadth_pct >= 30:
        breadth_label = "Weak"
    else:
        breadth_label = "Very Weak (Risk-Off)"

    # ── Regime Classification ──
    fg_val = fg.get("value", 50)

    if btc_trend == "Bullish" and alt_breadth_pct >= 60:
        regime = "🚀 Risk-On Bull Run"
        summary = (f"BTC is in a clear uptrend (7d: {btc_ret7:+.1f}%) and alt breadth is strong "
                   f"({alt_breadth_pct:.0f}% of alts above key EMAs). "
                   f"Fear & Greed at {fg_val} ({fg.get('label','?')}). "
                   f"This regime historically favors momentum longs, though vol is {vol_regime.lower()}.")
    elif btc_trend == "Bullish" and alt_breadth_pct < 60:
        regime = "₿ BTC-Dominated"
        summary = (f"BTC is trending up (7d: {btc_ret7:+.1f}%) but alt breadth is limited "
                   f"({alt_breadth_pct:.0f}%), suggesting capital concentration in BTC. "
                   f"Alts may lag or be volatile. Watch for breadth expansion as a rotation signal.")
    elif btc_trend == "Bearish" and alt_breadth_pct < 40:
        regime = "🔴 Risk-Off Bear"
        summary = (f"BTC is in a downtrend (7d: {btc_ret7:+.1f}%) and alt breadth is weak "
                   f"({alt_breadth_pct:.0f}%). F&G={fg_val} ({fg.get('label','?')}). "
                   f"This regime historically produces oversold bounces but sustained reversals are rare "
                   f"without BTC stabilizing first. Volatility: {vol_regime}.")
    elif vol_regime in ["Extreme", "High"] and btc_trend == "Neutral":
        regime = "⚡ High-Volatility Chop"
        summary = (f"BTC is directionless with elevated volatility (ATR rank: {atr_rank:.0f}th percentile). "
                   f"Alt breadth is {alt_breadth_pct:.0f}%, F&G={fg_val}. "
                   f"Expect whipsaws. Signals have lower predictive value in this regime.")
    else:
        regime = "↔️ Transitional / Mixed"
        summary = (f"BTC ({btc_trend}, 7d: {btc_ret7:+.1f}%) and alts (breadth: {alt_breadth_pct:.0f}%) "
                   f"are sending mixed signals. F&G={fg_val} ({fg.get('label','?')}). "
                   f"Treat individual coin signals as lower-confidence until a clearer macro trend emerges.")

    return {
        "regime":         regime,
        "btc_trend":      btc_trend,
        "btc_7d_pct":     round(btc_ret7, 2),
        "alt_breadth":    f"{alt_breadth_pct:.0f}% — {breadth_label}",
        "volatility":     vol_regime,
        "fg_value":       fg_val,
        "fg_label":       fg.get("label", "N/A"),
        "fg_trend":       fg.get("trend", "flat"),
        "summary":        summary,
    }


print("✅ Market regime detector defined")

## 9) Main Dashboard Builder

In [ ]:
def run_dashboard(
    coins     = COINS,
    quote     = QUOTE,
    timeframe = TIMEFRAME,
    lookback  = LOOKBACK_BARS,
    exchange  = EXCHANGE_ID,
    weights   = WEIGHTS,
):
    from zoneinfo import ZoneInfo
    cet = ZoneInfo("Europe/Zurich")
    print(f"\n{'='*70}")
    print(f" Advanced Crypto Dashboard v{VERSION} — {datetime.now(cet).strftime('%Y-%m-%d %H:%M CET')}")
    print(f" Exchange: {exchange} | Primary: {timeframe} | Confirmation: {TF_FAST}/{TF_SLOW}")
    print(f"{'='*70}\n")

    # ── Fetch auxiliary data ─────────────────────────────────────────
    print("📡 Fetching Fear & Greed Index...", end=" ")
    fg = fetch_fear_greed()
    print(f"{fg['value']} ({fg['label']}, 7d avg: {fg['avg7']}, trend: {fg['trend']})")

    print("📡 Fetching BTC Dominance...", end=" ")
    dominance = fetch_btc_dominance()
    print(f"{dominance['dominance']}% ({dominance['trend']})")

    print("📡 Fetching Funding Rates...", end=" ")
    funding_map = fetch_funding_rates(coins)
    available = sum(1 for v in funding_map.values() if v['bias'] != 'unavailable')
    print(f"{available}/{len(coins)} coins available")

    print("📡 Fetching News Sentiment...", end=" ")
    news_scores = fetch_news_sentiment(coins)
    print("done" if CRYPTOPANIC_KEY else "skipped (no API key)")

    # ── Fetch primary OHLCV ──────────────────────────────────────────
    data_map = {}
    for coin in coins:
        sym = f"{coin}/{quote}"
        print(f"📡 Fetching {sym} ({timeframe})...", end=" ")
        try:
            df = fetch_ohlcv(sym, timeframe=timeframe, limit=lookback, exchange_id=exchange)
            df = add_all_indicators(df)
            data_map[coin] = df
            print(f"{len(df)} bars, close: {df['close'].iloc[-1]:.6g}")
        except Exception as e:
            print(f"❌ ERROR: {e}")
            data_map[coin] = None

    btc_df = data_map.get("BTC")

    # ── Fetch multi-timeframe signals ────────────────────────────────
    print("\n📡 Fetching multi-timeframe signals...")
    mtf_map = {}
    for coin in coins:
        try:
            mtf_map[coin] = fetch_mtf_signals(coin, quote, exchange)
            tf_str = " | ".join(f"{tf}: {s:.0f}" for tf, s in mtf_map[coin].items())
            print(f"   {coin}: {tf_str}")
        except Exception:
            mtf_map[coin] = {}

    # ── Score each coin ──────────────────────────────────────────────
    rows = []
    score_details = {}

    for coin in coins:
        df = data_map.get(coin)
        if df is None or len(df) < 50:
            rows.append({"Coin": coin, "Score": "N/A", "Signal": "❓ No Data",
                         "Confidence": "—", "Key Driver": "—", "Agreement": "—",
                         "RSI": "—", "MACD ▲▼": "—", "Above EMA50": "—",
                         "Vol Ratio": "—", "Funding": "—", "MTF 1h/1d": "—",
                         "Divergence": "—"})
            continue

        scores = composite_score(
            df, btc_df, fg, coin,
            funding_map  = funding_map,
            mtf_signals  = mtf_map.get(coin, {}),
            dominance    = dominance,
            news_scores  = news_scores,
            weights      = weights,
        )
        signal = classify_signal(scores, coin)
        score_details[coin] = scores

        last = df.iloc[-1]
        fr   = funding_map.get(coin, {})
        mtf  = mtf_map.get(coin, {})
        vol_r = _safe_float(last.get('vol_ratio', 1.0))
        vol_capped = vol_r < VOL_CAP_STRONG and scores['composite'] > 50
        rows.append({
            "Coin":       coin,
            "Score":      f"{scores['composite']:.0f}/100" + (" 🔇" if vol_capped else ""),
            "Signal":     signal["direction"],
            "Confidence": signal["confidence"],
            "Key Driver": signal["key_driver"],
            "Agreement":  signal["agreement"],
            "RSI":        f"{_safe_float(last.get('rsi', 50)):.1f}",
            "MACD ▲▼":   "▲" if last.get("macd_expanding") else "▼",
            "EMA50":      "✅" if _safe_float(last.get("close",0)) > _safe_float(last.get("ema50",0)) else "❌",
            "Vol Ratio":  f"{_safe_float(last.get('vol_ratio',1)):.2f}x",
            "Funding":    f"{fr.get('rate', 0):+.3f}% ({fr.get('bias','?')}) [{fr.get('source','?')}]",
            "MTF 1h/1d":  f"{mtf.get(TF_FAST, 50):.0f}/{mtf.get(TF_SLOW, 50):.0f}",
            "Divergence":  signal["divergence"],
        })

    dashboard_df = pd.DataFrame(rows)

    def sort_key(row):
        s = row["Score"]
        try:
            return -float(s.split("/")[0])
        except Exception:
            return 0

    dashboard_df = dashboard_df.iloc[sorted(range(len(dashboard_df)), key=lambda i: sort_key(dashboard_df.iloc[i]))]
    dashboard_df = dashboard_df.reset_index(drop=True)

    regime = detect_market_regime(data_map, fg)

    # ── Bounce Quality Analysis ──────────────────────────────────────
    bounce_analysis = run_bounce_analysis(data_map, score_details, mtf_map, fg)

    return dashboard_df, score_details, regime, data_map, bounce_analysis, mtf_map, fg


print("✅ Dashboard v3 builder defined — ready to run!")

## 10) Run the Dashboard

▶️ Run this cell to fetch live data and generate the full analysis.

In [ ]:
# ── Run ──────────────────────────────────────────────────────────────
dashboard, score_details, regime, data_map, bounce_analysis, mtf_map, fg = run_dashboard()

# ── Print Market Regime ──────────────────────────────────────────────
print("\n" + "═"*70)
print("  MARKET REGIME SUMMARY")
print("═"*70)
print(f"  Regime:      {regime['regime']}")
print(f"  BTC Trend:   {regime['btc_trend']} (7d: {regime['btc_7d_pct']:+.2f}%)")
print(f"  Alt Breadth: {regime['alt_breadth']}")
print(f"  Volatility:  {regime['volatility']}")
print(f"  Fear & Greed: {regime['fg_value']} — {regime['fg_label']} (trend: {regime['fg_trend']})")
print()
print(f"  {regime['summary']}")
print("═"*70 + "\n")

# ── Print Dashboard Table ────────────────────────────────────────────
print("  COIN SIGNALS (sorted by composite score)")
print("  🔇 = score capped due to low volume")
print()
display(dashboard)
print()

# ── Print Bounce Quality Analysis ────────────────────────────────────
if bounce_analysis:
    print("\n" + "═"*70)
    print("  BOUNCE QUALITY ANALYSIS")
    print("  (Only shown for coins currently signalling Bullish or above)")
    print("═"*70)
    for coin, result in bounce_analysis.items():
        print(f"\n  {coin} — {result['quality']}  (Sustainability Score: {result['score']:.0f}/100, Confidence: {result['confidence']})")
        if result['flags']:
            print("  ⚠️  Red Flags:")
            for f in result['flags']:
                print(f"     • {f}")
        if result['green']:
            print("  ✅ Supporting Factors:")
            for g in result['green']:
                print(f"     • {g}")
        if result['warning']:
            print(f"\n  📋 Assessment: {result['warning']}")
        print("  " + "─"*66)
    print("═"*70 + "\n")

# ── Print Sub-Score Breakdown ────────────────────────────────────────
if score_details:
    print("\n" + "─"*80)
    print("  SUB-SCORE BREAKDOWN (0=Bearish, 50=Neutral, 100=Bullish)")
    print("─"*80)
    breakdown_rows = []
    for coin, scores in score_details.items():
        row = {"Coin": coin}
        for k in ["momentum","trend","volume","volatility","correlation","sentiment","funding","mtf"]:
            row[k.capitalize()] = f"{scores.get(k, 50):.0f}"
        row["COMPOSITE"] = f"{scores['composite']:.0f}"
        breakdown_rows.append(row)
    display(pd.DataFrame(breakdown_rows))

## 11) Per-Coin Deep Dive

Run this cell to inspect any specific coin in detail.

In [ ]:
# ── Change INSPECT_COIN to any coin in your list ─────────────────────
INSPECT_COIN = "BTC"

df = data_map.get(INSPECT_COIN)
if df is None:
    print(f"No data for {INSPECT_COIN}")
else:
    last  = df.iloc[-1]
    close = _safe_float(last.get("close", 0))

    print(f"\n{'='*60}")
    print(f"  {INSPECT_COIN} DEEP DIVE")
    print(f"{'='*60}")
    print(f"  Latest close:   {close:.6g}")
    print(f"  EMA9:   {_safe_float(last.get('ema9',0)):.6g}  | {'✅ above' if close > _safe_float(last.get('ema9',0)) else '❌ below'}")
    print(f"  EMA20:  {_safe_float(last.get('ema20',0)):.6g}  | {'✅ above' if close > _safe_float(last.get('ema20',0)) else '❌ below'}")
    print(f"  EMA50:  {_safe_float(last.get('ema50',0)):.6g}  | {'✅ above' if close > _safe_float(last.get('ema50',0)) else '❌ below'}")
    print(f"  EMA100: {_safe_float(last.get('ema100',0)):.6g}  | {'✅ above' if close > _safe_float(last.get('ema100',0)) else '❌ below'}")
    print(f"  EMA200: {_safe_float(last.get('ema200',0)):.6g}  | {'✅ above' if close > _safe_float(last.get('ema200',0)) else '❌ below'}")
    print()
    print(f"  RSI(14):       {_safe_float(last.get('rsi',50)):.1f}")
    print(f"  MACD:          {_safe_float(last.get('macd',0)):+.6g}")
    print(f"  MACD Signal:   {_safe_float(last.get('macd_signal',0)):+.6g}")
    print(f"  MACD Hist:     {_safe_float(last.get('macd_hist',0)):+.6g} ({'▲ expanding' if last.get('macd_expanding') else '▼ contracting'})")
    print(f"  Stoch %K:      {_safe_float(last.get('stoch_k',50)):.1f}")
    print(f"  Stoch %D:      {_safe_float(last.get('stoch_d',50)):.1f}")
    print(f"  ADX:           {_safe_float(last.get('adx',20)):.1f}  (+DI: {_safe_float(last.get('plus_di',25)):.1f}, -DI: {_safe_float(last.get('minus_di',25)):.1f})")
    print(f"  BB %B:         {_safe_float(last.get('bb_pct',0.5)):.2f}  (0=lower band, 0.5=mid, 1=upper)")
    print(f"  BB Bandwidth:  {_safe_float(last.get('bb_bw',0)):.4f}  (rank: {_safe_float(last.get('bb_bw_pct',50)):.0f}th pct)")
    print(f"  ATR%:          {_safe_float(last.get('atr_pct',0)):.2f}%  (rank: {_safe_float(last.get('atr_pct_rank',50)):.0f}th pct)")
    print(f"  OBV trend:     {'📈 Accumulation' if last.get('obv_trend') else '📉 Distribution'}")
    print(f"  Volume ratio:  {_safe_float(last.get('vol_ratio',1)):.2f}x 20-bar avg")
    print(f"  VWAP:          {_safe_float(last.get('vwap',close)):.6g}  | {'✅ above' if close > _safe_float(last.get('vwap',close)) else '❌ below'}")
    print()
    print(f"  Returns:")
    print(f"    1 bar:  {_safe_float(last.get('ret_1',0))*100:+.2f}%")
    print(f"    ~1 day: {_safe_float(last.get('ret_6',0))*100:+.2f}%")
    print(f"    ~3 day: {_safe_float(last.get('ret_18',0))*100:+.2f}%")
    print(f"    ~7 day: {_safe_float(last.get('ret_42',0))*100:+.2f}%")

    if INSPECT_COIN in score_details:
        s = score_details[INSPECT_COIN]
        print()
        print(f"  Score breakdown:")
        for k, v in s.items():
            bar = "█" * int(v // 10) + "░" * (10 - int(v // 10))
            print(f"    {k:<12} {bar} {v:.0f}/100")

## 12) Signal Accuracy Backtest (Simple)

A rolling backtest to check how often a composite score > 65 or < 35 correctly predicted the direction over the next N bars.

In [ ]:
# ── Backtest Parameters ───────────────────────────────────────────────
BACKTEST_COIN   = "BTC"
FORWARD_BARS    = 18        # bars to look ahead (~3 days on 4h)
BULL_THRESHOLD  = 65
BEAR_THRESHOLD  = 35
WARMUP          = 220       # bars needed for all indicators to stabilize

def backtest_signal(
    coin          = BACKTEST_COIN,
    forward_bars  = FORWARD_BARS,
    bull_thresh   = BULL_THRESHOLD,
    bear_thresh   = BEAR_THRESHOLD,
):
    df_full = data_map.get(coin)
    if df_full is None or len(df_full) < WARMUP + forward_bars + 5:
        print(f"Not enough data to backtest {coin}")
        return None

    btc_df = data_map.get("BTC")
    fg_neutral = {"value": 50, "label": "Neutral", "avg7": 50, "trend": "flat"}

    results = []
    for i in range(WARMUP, len(df_full) - forward_bars):
        slice_df = df_full.iloc[:i+1]
        btc_slice = btc_df.iloc[:i+1] if btc_df is not None else None

        try:
            scores = composite_score(slice_df, btc_slice, fg_neutral, coin)
        except Exception:
            continue

        c_now     = float(df_full["close"].iloc[i])
        c_future  = float(df_full["close"].iloc[i + forward_bars])
        fwd_ret   = (c_future - c_now) / c_now
        composite = scores["composite"]

        if composite >= bull_thresh:
            signal = "Bullish"
            correct = fwd_ret > 0
        elif composite <= bear_thresh:
            signal = "Bearish"
            correct = fwd_ret < 0
        else:
            continue   # neutral signals — skip

        results.append({
            "ts":        df_full.index[i],
            "composite": composite,
            "signal":    signal,
            "fwd_ret":   fwd_ret,
            "correct":   correct,
        })

    if not results:
        print("No signals generated in backtest window.")
        return None

    res_df = pd.DataFrame(results)
    bull_df = res_df[res_df["signal"] == "Bullish"]
    bear_df = res_df[res_df["signal"] == "Bearish"]

    print(f"\n{'='*60}")
    print(f"  BACKTEST: {coin} | Forward: {forward_bars} bars (~{forward_bars*4}h)")
    print(f"  Period: {df_full.index[WARMUP].date()} → {df_full.index[-forward_bars-1].date()}")
    print(f"{'='*60}")
    print(f"  Total signals:       {len(res_df)}")
    print(f"  Bullish signals:     {len(bull_df)}  | Accuracy: {bull_df['correct'].mean()*100:.1f}%" if len(bull_df) else "  Bullish signals:  0")
    print(f"  Bearish signals:     {len(bear_df)}  | Accuracy: {bear_df['correct'].mean()*100:.1f}%" if len(bear_df) else "  Bearish signals:  0")
    print(f"  Overall accuracy:    {res_df['correct'].mean()*100:.1f}%")
    print(f"  Avg fwd return (bull signals): {bull_df['fwd_ret'].mean()*100:+.2f}%" if len(bull_df) else "")
    print(f"  Avg fwd return (bear signals): {bear_df['fwd_ret'].mean()*100:+.2f}%" if len(bear_df) else "")
    print()
    print("  ⚠️  Backtest is in-sample (same data used for signals). "
          "Results are indicative only and do not guarantee future performance.")
    return res_df


backtest_results = backtest_signal()

## 13) Notes on Accuracy & Limitations

### What makes this notebook more accurate than the original?

| Dimension | Original | This Notebook |
|---|---|---|
| Indicators | RSI, EMA×3, Volume | RSI, MACD, BB, ADX, Stochastic, ATR, OBV, VWAP, EMA×5 |
| Scoring | Hand-coded integer points | Weighted ensemble of 6 continuous sub-scores |
| Volume analysis | Simple spike check | OBV trend, buy/sell pressure, vol momentum |
| Market context | Basic BTC/alt comparison | Regime detection (5 regimes), BTC correlation, beta |
| Sentiment | None | Live Fear & Greed Index (contrarian) |
| Signal quality | Binary (Yes/No/Developing) | Continuous 0–100 with confidence level |
| Backtest | None | Rolling accuracy check |

### Why no ML model?

- **Crypto markets are non-stationary**: a model trained on 2021 data performs poorly in 2025. Indicator-based rules adapt implicitly to current volatility regimes.
- **Overfitting risk**: with ~360 bars of training data, an ML model will memorize noise. A structured indicator ensemble is more robust out-of-sample.
- **Interpretability**: you can see exactly why a coin is scored high or low.

### Known limitations

1. **Fear & Greed only reflects BTC/crypto overall** — coin-specific sentiment (Twitter, Reddit) is not included.
2. **On-chain data** (exchange flows, whale activity, funding rates) would significantly improve accuracy but requires paid APIs (Glassnode, Santiment, CoinGlass).
3. **Macro factors** (DXY, US10Y, SP500 correlation) are not fetched in real-time — add if you have access to a financial data API.
4. **The backtest is in-sample** — signals are generated on the same data the indicators were computed on. A proper walk-forward backtest would require more data.

### How to improve further

- Add CoinGlass **funding rates** + **open interest** (perpetual futures pressure)
- Add exchange **net flows** (coins moving to/from exchanges) via Glassnode API
- Add **on-chain active addresses** as a demand proxy
- Use **multiple timeframe confirmation** (e.g., 1h + 4h + daily must agree)
- Subscribe to a news/social sentiment API for coin-specific NLP scoring

## 14) Copy for Analysis

Run this cell after Section 10 to generate a compact summary for pasting to Claude.
Copy everything between the `=== COPY START ===` and `=== COPY END ===` markers.

In [ ]:
def build_analysis_text(
    dashboard, score_details, regime, bounce_analysis, mtf_map, fg
):
    """
    Builds the compact analysis text for pasting to Claude.
    Returns the full string.
    """
    from zoneinfo import ZoneInfo
    import io
    buf = io.StringIO()

    def p(*args, **kwargs):
        print(*args, **kwargs, file=buf)

    cet = ZoneInfo("Europe/Zurich")
    ts  = datetime.now(cet).strftime('%Y-%m-%d %H:%M CET')

    # ── Header ───────────────────────────────────────────────────────
    p(f"Dashboard v{VERSION} | {ts}")
    p(f"F&G: {fg['value']} ({fg['label']}, trend: {fg['trend']}) | "
      f"BTC Dom: {regime.get('btc_7d_pct', 0):+.1f}% 7d | "
      f"Regime: {regime['regime']}")
    p()

    # ── Signals ──────────────────────────────────────────────────────
    p("SIGNALS:")
    p(f"{'Coin':<6} {'Score':>7}  {'Signal':<22} {'RSI':>5}  "
      f"{'MACD':>4}  {'EMA50':>5}  {'Vol':>5}  {'MTF':>7}  "
      f"{'Funding':>22}  Divergence")
    p("-" * 118)
    for _, row in dashboard.iterrows():
        coin  = str(row.get("Coin", "?"))
        score = str(row.get("Score", "?"))
        sig   = str(row.get("Signal", "?"))
        rsi   = str(row.get("RSI", "?"))
        macd  = str(row.get("MACD \u25b2\u25bc", "?"))
        ema   = str(row.get("EMA50", "?"))
        vol   = str(row.get("Vol Ratio", "?"))
        mtf   = str(row.get("MTF 1h/1d", "?"))
        fund  = str(row.get("Funding", "?"))
        fund_short = fund.split(" [")[0] if " [" in fund else fund
        div   = str(row.get("Divergence", "\u2014"))
        p(f"{coin:<6} {score:>7}  {sig:<22} {rsi:>5}  "
          f"{macd:>4}  {ema:>5}  {vol:>5}  {mtf:>7}  "
          f"{fund_short:>22}  {div}")
    p()

    # ── Sub-scores ───────────────────────────────────────────────────
    p("SUB-SCORES (0=Bear 50=Neutral 100=Bull):")
    keys = ["momentum","trend","volume","volatility",
            "correlation","sentiment","funding","mtf","composite"]
    hdr = f"{'Coin':<6}  " + "  ".join(f"{k[:4]:>4}" for k in keys)
    p(hdr)
    p("-" * len(hdr))
    for coin, scores in score_details.items():
        vals = "  ".join(f"{scores.get(k, 50):>4.0f}" for k in keys)
        p(f"{coin:<6}  {vals}")
    p()

    # ── Bounce quality ───────────────────────────────────────────────
    if bounce_analysis:
        p("BOUNCE QUALITY:")
        for coin, b in bounce_analysis.items():
            p(f"  {coin}: {b['quality']} ({b['score']:.0f}/100, {b['confidence']})")
            for flag in b["flags"]:
                p(f"    \u26a0 {flag}")
            for g in b["green"]:
                p(f"    \u2713 {g}")
            if b["warning"]:
                p(f"    \u2192 {b['warning']}")
        p()

    # ── Regime ───────────────────────────────────────────────────────
    p("REGIME:")
    p(f"  {regime['summary']}")

    return buf.getvalue()


def show_analysis_with_copy_button(
    dashboard, score_details, regime, bounce_analysis, mtf_map, fg
):
    """
    Displays the analysis text and a 📋 Copy button.
    Clicking the button copies the text to the clipboard via JavaScript.
    Falls back to plain print if ipywidgets is unavailable.
    """
    text = build_analysis_text(
        dashboard, score_details, regime, bounce_analysis, mtf_map, fg
    )

    try:
        import ipywidgets as widgets
        from IPython.display import display, Javascript

        # Text area showing the content
        ta = widgets.Textarea(
            value=text,
            layout=widgets.Layout(
                width="100%",
                height="420px",
                font_family="monospace"
            )
        )

        # Copy button
        btn = widgets.Button(
            description="Copy to clipboard",
            button_style="success",
            layout=widgets.Layout(width="220px", height="36px")
        )
        status = widgets.Label(value="")

        def on_copy(b):
            # Escape backticks and backslashes for JS template literal
            safe = text.replace("\\", "\\\\").replace("`", "\\`")
            display(Javascript(f"""
                navigator.clipboard.writeText(`{safe}`).then(
                    () => {{ IPython.notebook.kernel.execute('__copy_ok = True') }},
                    () => {{ IPython.notebook.kernel.execute('__copy_ok = False') }}
                );
            """))
            status.value = "Copied!"

        btn.on_click(on_copy)

        display(widgets.VBox([
            widgets.HBox([btn, status]),
            ta
        ]))

    except ImportError:
        # ipywidgets not available — plain print fallback
        print("=" * 60)
        print("=== COPY START ===")
        print("=" * 60)
        print(text)
        print("=" * 60)
        print("=== COPY END ===")
        print("=" * 60)


# ── Run after Section 10 ─────────────────────────────────────────────
show_analysis_with_copy_button(
    dashboard, score_details, regime, bounce_analysis, mtf_map, fg
)